In [ ]:
!pip install torch-geometric

In [ ]:
import sys, torch
import torch_geometric
print(f'Python      : {sys.version.split()[0]}')
print(f'PyTorch     : {torch.__version__}')
print(f'PyG         : {torch_geometric.__version__}')
print(f'CUDA        : {torch.version.cuda}')
print(f'Device      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# pinned versions for reproducibility
# torch==2.1.0  torch-geometric==2.4.0  numpy==1.26.0

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import numpy as np

In [ ]:
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]
print(f'Nodes: {data.num_nodes}, Features: {data.num_node_features}, Classes: {dataset.num_classes}')

In [ ]:
def make_low_label_split(data, n_per_class=5, seed=42):
    torch.manual_seed(seed)
    n_classes = data.y.max().item() + 1
    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    val_mask   = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask  = torch.zeros(data.num_nodes, dtype=torch.bool)

    for c in range(n_classes):
        idx = (data.y == c).nonzero(as_tuple=True)[0]
        perm = idx[torch.randperm(len(idx))]
        train_mask[perm[:n_per_class]] = True         # 5 labeled
        val_mask[perm[n_per_class:n_per_class+50]] = True  # 50 val
        test_mask[perm[n_per_class+50:]] = True        # rest test

    data.train_mask = train_mask
    data.val_mask   = val_mask
    data.test_mask  = test_mask
    return data

data = make_low_label_split(data)
print(f'Train: {data.train_mask.sum()}, Val: {data.val_mask.sum()}, Test: {data.test_mask.sum()}')

# Graph Encoder

In [ ]:
class GCNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x  # node embeddings: [N, out_dim]

# Pretrain

In [ ]:
class DGIDiscriminator(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.bilinear = nn.Bilinear(dim, dim, 1)

    def forward(self, node_emb, summary):
        # node_emb: [N, dim], summary: [dim] -> score: [N]
        summary = summary.unsqueeze(0).expand_as(node_emb)
        return self.bilinear(node_emb, summary).squeeze()

In [ ]:
def pretrain_dgi(encoder, disc, data, epochs=200, lr=1e-3):
    params = list(encoder.parameters()) + list(disc.parameters())
    opt = torch.optim.Adam(params, lr=lr)
    bce = nn.BCEWithLogitsLoss()

    for ep in range(epochs):
        encoder.train(); disc.train()
        opt.zero_grad()

        # positive: real node embeddings
        h = encoder(data.x, data.edge_index)  # [N, dim]
        summary = torch.sigmoid(h.mean(0))    # [dim] graph-level

        # negative: corrupt node features (shuffle rows)
        perm = torch.randperm(data.num_nodes)
        h_neg = encoder(data.x[perm], data.edge_index)

        pos_scores = disc(h, summary)
        neg_scores = disc(h_neg, summary)

        labels_pos = torch.ones(data.num_nodes)
        labels_neg = torch.zeros(data.num_nodes)

        loss = bce(pos_scores, labels_pos) + bce(neg_scores, labels_neg)
        loss.backward()
        opt.step()

        if (ep + 1) % 50 == 0:
            print(f'  Pretrain epoch {ep+1}/{epochs}  loss={loss.item():.4f}')

    return encoder

In [ ]:
HIDDEN = 256
EMB_DIM = 128
N_CLASSES = dataset.num_classes

encoder = GCNEncoder(data.num_node_features, HIDDEN, EMB_DIM)
disc = DGIDiscriminator(EMB_DIM)

print('Pretraining encoder with DGI...')
encoder = pretrain_dgi(encoder, disc, data, epochs=200)
print('Pretraining done.')

# Freeze Encoder

In [ ]:
for param in encoder.parameters():
    param.requires_grad = False
encoder.eval()
print('Encoder frozen. Trainable params:', sum(p.numel() for p in encoder.parameters() if p.requires_grad))

In [ ]:
# X_prompt = X + p
class AdditivePrompt(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        # self.p = nn.Parameter(torch.zeros(feat_dim))  # single trainable vector
        self.alpha = nn.Parameter(torch.ones(feat_dim))
        self.beta  = nn.Parameter(torch.zeros(feat_dim))

    def forward(self, x):
        # return x + self.p  # broadcast: [N, F] + [F] -> [N, F]
        return self.alpha * x + self.beta

In [ ]:
class ClassifierHead(nn.Module):
    def __init__(self, emb_dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(emb_dim, n_classes)

    def forward(self, h):
        return self.fc(h)

In [ ]:
def train_downstream(encoder, data, prompt=None, epochs=200, lr=1e-2):
    head = ClassifierHead(EMB_DIM, N_CLASSES)

    # trainable params = prompt (if any) + head
    params = list(head.parameters())
    if prompt is not None:
        params += list(prompt.parameters())

    opt = torch.optim.Adam(params, lr=lr, weight_decay=5e-4)
    best_val_acc, best_test_acc = 0.0, 0.0
    val_curve = []

    for ep in range(epochs):
        head.train()
        if prompt: prompt.train()
        opt.zero_grad()

        # if prompt exists, grad flows through it; encoder stays frozen
        if prompt:
            x = prompt(data.x)
            h = encoder(x, data.edge_index)
        else:
            with torch.no_grad():
                h = encoder(data.x, data.edge_index)

        logits = head(h)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()

        # eval
        head.eval()
        if prompt: prompt.eval()
        with torch.no_grad():
            x = prompt(data.x) if prompt else data.x
            h = encoder(x, data.edge_index)
            preds = head(h).argmax(1)
            val_acc  = (preds[data.val_mask]  == data.y[data.val_mask]).float().mean().item()
            test_acc = (preds[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        val_curve.append(val_acc) 
        
        if val_acc > best_val_acc:
            best_val_acc  = val_acc
            best_test_acc = test_acc

    return best_val_acc, best_test_acc, val_curve 

In [ ]:
print('=== Baseline (no prompt) ===')
val_b, test_b, _ = train_downstream(encoder, data, prompt=None)
print(f'Val acc: {val_b:.4f}  Test acc: {test_b:.4f}')

In [ ]:
prompt = AdditivePrompt(data.num_node_features)

prompt_params = sum(p.numel() for p in prompt.parameters())
head_params   = EMB_DIM * N_CLASSES + N_CLASSES  # linear layer
encoder_params = sum(p.numel() for p in encoder.parameters())
print(f'Encoder params (frozen): {encoder_params}')
print(f'Prompt params: {prompt_params}')
print(f'Classifier head params: {head_params}')

print('\n=== Prompt-tuned ===')
val_p, test_p, _ = train_downstream(encoder, data, prompt=prompt)
print(f'Val acc: {val_p:.4f}  Test acc: {test_p:.4f}')

In [ ]:
print('\n--- Results ---')
print(f'Pretraining method : DGI')
print(f'Prompt type        : Additive feature prompt')
print(f'Prompt params      : {data.num_node_features}')
print(f'Trainable (baseline): {head_params}')
print(f'Trainable (prompt)  : {data.num_node_features + head_params}')
print(f'Baseline test acc  : {test_b:.4f}')
print(f'Prompt test acc    : {test_p:.4f}')
print(f'Delta              : {test_p - test_b:+.4f}')

In [ ]:
def run_seeds(encoder, data, use_prompt, seeds=[0,1,2]):
    results = []
    for s in seeds:
        torch.manual_seed(s)
        # rebuild split with this seed so label selection varies
        d = make_low_label_split(data.clone(), seed=s)
        prompt = AdditivePrompt(d.num_node_features) if use_prompt else None
        _, test_acc, _ = train_downstream(encoder, d, prompt=prompt)
        results.append(test_acc)
    arr = np.array(results)
    return arr.mean(), arr.std()

In [ ]:
print('Running baseline over 3 seeds...')
mean_b, std_b = run_seeds(encoder, data, use_prompt=False)

print('Running prompt-tuned over 3 seeds...')
mean_p, std_p = run_seeds(encoder, data, use_prompt=True)

In [ ]:
prompt_params = data.num_node_features
head_params   = EMB_DIM * N_CLASSES + N_CLASSES

print('--- Results (mean ± std over 3 seeds) ---')
print(f'Pretraining        : DGI')
print(f'Prompt type        : Additive feature prompt')
print(f'Encoder params (frozen)  : {sum(p.numel() for p in encoder.parameters())}')
print(f'Prompt params            : {prompt_params}')
print(f'Head params              : {head_params}')
print(f'Baseline  test acc : {mean_b:.4f} ± {std_b:.4f}')
print(f'Prompted  test acc : {mean_p:.4f} ± {std_p:.4f}')
print(f'Delta              : {mean_p - mean_b:+.4f}')

# Ablations

In [ ]:
# Ablation 1: does pretraining matter?
# prompt on a RANDOM (untrained) encoder vs prompt on pretrained encoder
# if pretraining matters, random+prompt should lag behind pretrained+prompt

random_encoder = GCNEncoder(data.num_node_features, HIDDEN, EMB_DIM)
for p in random_encoder.parameters():
    p.requires_grad = False

print('Ablation 1 — random encoder + prompt (no pretraining)...')
mean_rand, std_rand = run_seeds(random_encoder, data, use_prompt=True)
print(f'  Test acc: {mean_rand:.4f} ± {std_rand:.4f}')

In [ ]:
# Ablation 2: does freezing matter?
# unfreeze the encoder and train everything end-to-end (standard fine-tuning)
# compare against frozen+prompt to isolate the value of keeping encoder fixed
import copy

def run_seeds_finetune(encoder_init, data, seeds=[0,1,2]):
    results = []
    for s in seeds:
        torch.manual_seed(s)
        d = make_low_label_split(data.clone(), seed=s)

        # fresh copy so we don't corrupt the pretrained weights
        enc_copy = copy.deepcopy(encoder_init)
        for p in enc_copy.parameters():
            p.requires_grad = True          # unfreeze

        head = ClassifierHead(EMB_DIM, N_CLASSES)
        opt  = torch.optim.Adam(
            list(enc_copy.parameters()) + list(head.parameters()), lr=1e-2, weight_decay=5e-4
        )
        best_val, best_test = 0.0, 0.0
        for ep in range(200):
            enc_copy.train(); head.train()
            opt.zero_grad()
            h = enc_copy(d.x, d.edge_index)
            loss = F.cross_entropy(head(h)[d.train_mask], d.y[d.train_mask])
            loss.backward(); opt.step()
            enc_copy.eval(); head.eval()
            with torch.no_grad():
                preds = head(enc_copy(d.x, d.edge_index)).argmax(1)
                val_acc  = (preds[d.val_mask]  == d.y[d.val_mask]).float().mean().item()
                test_acc = (preds[d.test_mask] == d.y[d.test_mask]).float().mean().item()
            if val_acc > best_val:
                best_val, best_test = val_acc, test_acc
        results.append(best_test)
    arr = np.array(results)
    return arr.mean(), arr.std()

print('Ablation 2 — unfrozen encoder (standard fine-tuning)...')
mean_ft, std_ft = run_seeds_finetune(encoder, data)
print(f'  Test acc: {mean_ft:.4f} ± {std_ft:.4f}')

In [ ]:
# Ablation 3: virtual node prompt
# adds K trainable nodes connected to all original nodes
# encoder runs on augmented graph; virtual embeddings discarded after

class VirtualNodePrompt(nn.Module):
    def __init__(self, feat_dim, k=3):
        super().__init__()
        self.k = k
        self.virtual_feats = nn.Parameter(torch.randn(k, feat_dim) * 0.01)

    def augment(self, x, edge_index):
        N = x.size(0)
        x_aug = torch.cat([x, self.virtual_feats], dim=0)   # [N+k, F]

        # bidirectional edges: each virtual node <-> all original nodes
        orig = torch.arange(N)
        virt = torch.arange(N, N + self.k)
        src  = virt.repeat_interleave(N)
        dst  = orig.repeat(self.k)
        new_edges  = torch.stack([torch.cat([src, dst]),
                                   torch.cat([dst, src])], dim=0)
        edge_aug = torch.cat([edge_index, new_edges], dim=1)
        return x_aug, edge_aug, N

def train_downstream_virtual(encoder, data, prompt, epochs=200, lr=1e-2):
    head   = ClassifierHead(EMB_DIM, N_CLASSES)
    params = list(head.parameters()) + list(prompt.parameters())
    opt    = torch.optim.Adam(params, lr=lr, weight_decay=5e-4)
    best_val_acc, best_test_acc = 0.0, 0.0

    for ep in range(epochs):
        head.train(); prompt.train()
        opt.zero_grad()

        x_aug, edge_aug, N = prompt.augment(data.x, data.edge_index)
        h = encoder(x_aug, edge_aug)[:N]     # discard virtual node embeddings

        loss = F.cross_entropy(head(h)[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()

        head.eval(); prompt.eval()
        with torch.no_grad():
            x_aug, edge_aug, N = prompt.augment(data.x, data.edge_index)
            h     = encoder(x_aug, edge_aug)[:N]
            preds = head(h).argmax(1)
            val_acc  = (preds[data.val_mask]  == data.y[data.val_mask]).float().mean().item()
            test_acc = (preds[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc  = val_acc
            best_test_acc = test_acc

    return best_val_acc, best_test_acc

def run_seeds_virtual(encoder, data, seeds=[0,1,2]):
    results = []
    for s in seeds:
        torch.manual_seed(s)
        d      = make_low_label_split(data.clone(), seed=s)
        prompt = VirtualNodePrompt(d.num_node_features, k=3)
        _, test_acc = train_downstream_virtual(encoder, d, prompt)
        results.append(test_acc)
    arr = np.array(results)
    return arr.mean(), arr.std()

print('Ablation 3 — virtual node prompt...')
mean_virt, std_virt = run_seeds_virtual(encoder, data)
print(f'  Test acc: {mean_virt:.4f} ± {std_virt:.4f}')

In [ ]:
from torch_geometric.nn import SAGEConv

class SAGEEncoder(nn.Module):
    def __init__(self, in_dim, hid, out):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hid)
        self.conv2 = SAGEConv(hid, out)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

In [ ]:
from torch_geometric.nn import GINConv

class GINEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        # GIN uses a trainable MLP inside each conv
        mlp1 = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        mlp2 = nn.Sequential(nn.Linear(hidden_dim, out_dim), nn.ReLU(), nn.Linear(out_dim, out_dim))
        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.bn2(self.conv2(x, edge_index))
        return x  # [N, out_dim]

In [ ]:
# Ablation 4: does pretraining model matter?
# try different encoders: SAGEEncoder, GINEncoder
for enc_name in ("SAGE", "GIN"):
    if enc_name == "SAGE":
        encoder   = SAGEEncoder(data.num_node_features, HIDDEN, EMB_DIM)
    elif enc_name == "GIN":
        encoder   = GINEncoder(data.num_node_features, HIDDEN, EMB_DIM)
    disc = DGIDiscriminator(EMB_DIM)
    encoder = pretrain_dgi(encoder, disc, data, epochs=200)
    
    for p in random_encoder.parameters():
        p.requires_grad = False
    encoder.eval()
    
    print(f'Ablation 4 — {enc_name} encoder + prompt...')
    mean_rand, std_rand = run_seeds(encoder, data, use_prompt=True)
    print(f'  Test acc: {mean_rand:.4f} ± {std_rand:.4f}')

In [ ]:
print('--- Results (mean ± std over 3 seeds) ---')
print(f'Pretraining              : DGI')
print(f'Prompt type (main)       : Additive feature prompt')
print(f'Encoder params (frozen)  : {sum(p.numel() for p in encoder.parameters())}')
print(f'Prompt params (additive) : {data.num_node_features}')
print(f'Prompt params (virtual)  : {3 * data.num_node_features}')
print()
print(f'Baseline  (no prompt)    : {mean_b:.4f} ± {std_b:.4f}')
print(f'Prompted  (additive)     : {mean_p:.4f} ± {std_p:.4f}')
print()
print(f'Ablation 1 — random enc  : {mean_rand:.4f} ± {std_rand:.4f}')
print(f'Ablation 2 — unfrozen enc: {mean_ft:.4f} ± {std_ft:.4f}')
print(f'Ablation 3 — virtual     : {mean_virt:.4f} ± {std_virt:.4f}')

In [ ]:
def run_pipeline(dataset_name, HIDDEN = 256, EMB_DIM = 128):
    dataset = Planetoid(root='data', name=dataset_name)
    data = dataset[0]
    print(f'Nodes: {data.num_nodes}, Features: {data.num_node_features}, Classes: {dataset.num_classes}')
    
    data = make_low_label_split(data)
    print(f'Train: {data.train_mask.sum()}, Val: {data.val_mask.sum()}, Test: {data.test_mask.sum()}')

    N_CLASSES = dataset.num_classes
    encoder = GCNEncoder(data.num_node_features, HIDDEN, EMB_DIM)
    disc = DGIDiscriminator(EMB_DIM)
    
    print('Pretraining encoder with DGI...')
    encoder = pretrain_dgi(encoder, disc, data, epochs=200)
    print('Pretraining done.')
    
    for param in encoder.parameters():
        param.requires_grad = False
    encoder.eval()
    print('Encoder frozen. Trainable params:', sum(p.numel() for p in encoder.parameters() if p.requires_grad))

    print('Running baseline over 3 seeds...')
    mean_b, std_b = run_seeds(encoder, data, use_prompt=False)
    
    print('Running prompt-tuned over 3 seeds...')

    mean_p, std_p = run_seeds(encoder, data, use_prompt=True)

    return mean_b, std_b, mean_p, std_p

In [ ]:
import pandas as pd

rows = []
for ds in ["Cora", "CiteSeer", "PubMed"]:
    print(f"Running {ds}...")
    mean_b, std_b, mean_p, std_p = run_pipeline(ds)
    rows.append({
        "Dataset": ds,
        "Baseline": f"{mean_b:.4f} ± {std_b:.4f}",
        "Prompt": f"{mean_p:.4f} ± {std_p:.4f}",
        "Delta": f"{mean_p - mean_b:+.4f}"
    })
results_df = pd.DataFrame(rows)
print(results_df)